In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
from pathlib import Path
import os
import yaml

os.chdir("C:/Users/ASRENOVIN/Desktop/spatial-foundation-graph-transformer")

with open("configs/default.yaml") as f:
    cfg = yaml.safe_load(f)

DATA_DIR = Path(cfg['paths']['data_raw']) / 'visium_breast_cancer'

print("Data dir:", DATA_DIR.exists())
print("Files found:")
for f in sorted(DATA_DIR.rglob('*')):
    print(" ", f.relative_to(DATA_DIR))



Data dir: True
Files found:
  spatial
  spatial\aligned_fiducials.jpg
  spatial\detected_tissue_image.jpg
  spatial\scalefactors_json.json
  spatial\tissue_hires_image.png
  spatial\tissue_lowres_image.png
  spatial\tissue_positions_list.csv
  V1_Breast_Cancer_Block_A_Section_1_filtered_feature_bc_matrix.h5
  V1_Breast_Cancer_Block_A_Section_1_raw_feature_bc_matrix.h5
  V1_Breast_Cancer_Block_A_Section_1_spatial.tar.gz


In [5]:
import scanpy as sc

adata = sc.read_visium(
    path=DATA_DIR,
    count_file="V1_Breast_Cancer_Block_A_Section_1_filtered_feature_bc_matrix.h5",
    load_images=True,
)

adata.var_names_make_unique()

print(adata)
print("\nSpots: ", adata.n_obs)
print("Genes: ", adata.n_vars)
print("Spatial keys: ", list(adata.obsm.keys()))
print("Image keys: ", list(adata.uns.get("spatial", {}).keys()))

C:\Users\ASRENOVIN\AppData\Local\Temp\ipykernel_9384\1638417183.py:3: FutureWarning: Use `squidpy.read.visium` instead.
  adata = sc.read_visium(
C:\ProgramData\miniconda3\envs\sfgt\lib\site-packages\anndata\_core\anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
C:\ProgramData\miniconda3\envs\sfgt\lib\site-packages\anndata\_core\anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


AnnData object with n_obs × n_vars = 3798 × 36601
    obs: 'in_tissue', 'array_row', 'array_col'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatial'
    obsm: 'spatial'

Spots:  3798
Genes:  36601
Spatial keys:  ['spatial']
Image keys:  ['V1_Breast_Cancer_Block_A_Section_1']


In [8]:
import squidpy as sq

adata = sq.read.visium(
    path=DATA_DIR,
    counts_file="V1_Breast_Cancer_Block_A_Section_1_filtered_feature_bc_matrix.h5",
)

adata.var_names_make_unique()

print(adata)
print("\nSpots: ", adata.n_obs)
print("Genes: ", adata.n_vars)
print("Spatial keys: ", list(adata.obsm.keys()))
print("Image keys: ", list(adata.uns.get("spatial", {}).keys()))

C:\ProgramData\miniconda3\envs\sfgt\lib\site-packages\anndata\_core\anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
C:\ProgramData\miniconda3\envs\sfgt\lib\site-packages\anndata\_core\anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


AnnData object with n_obs × n_vars = 3798 × 36601
    obs: 'in_tissue', 'array_row', 'array_col'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatial'
    obsm: 'spatial'

Spots:  3798
Genes:  36601
Spatial keys:  ['spatial']
Image keys:  ['V1_Breast_Cancer_Block_A_Section_1']


In [10]:
# save raw object
PROCESSED_DIR = Path(cfg["paths"]["data_processed"])
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

adata.write_h5ad(PROCESSED_DIR / "adata_raw.h5ad")

print("Saved to:", PROCESSED_DIR / "adata_raw.h5ad")
print("File size:", round(os.path.getsize(PROCESSED_DIR / "adata_raw.h5ad") / 1e6, 2), "MB")

Saved to: data\processed\adata_raw.h5ad
File size: 187.83 MB


In [11]:
visium_loader_code = '''from __future__ import annotations
from pathlib import Path
from typing import Any
import squidpy as sq
import anndata as ad


class VisiumDataset:
    """Loads 10x Visium data from local files into AnnData."""

    def __init__(self, cfg: dict[str, Any]) -> None:
        self.data_dir = Path(cfg["paths"]["data_raw"]) / "visium_breast_cancer"
        self.counts_file = "V1_Breast_Cancer_Block_A_Section_1_filtered_feature_bc_matrix.h5"
        self.processed_path = Path(cfg["paths"]["data_processed"]) / "adata_raw.h5ad"

    def load(self, force_reload: bool = False) -> ad.AnnData:
        """Return cached h5ad if available, otherwise read from raw files."""
        if self.processed_path.exists() and not force_reload:
            print(f"Loading cached AnnData from {self.processed_path}")
            return ad.read_h5ad(self.processed_path)
        print(f"Reading Visium data from {self.data_dir}")
        adata = sq.read.visium(path=self.data_dir, counts_file=self.counts_file)
        adata.var_names_make_unique()
        self.processed_path.parent.mkdir(parents=True, exist_ok=True)
        adata.write_h5ad(self.processed_path)
        print(f"Saved to {self.processed_path}")
        return adata


def load_visium(cfg: dict[str, Any], force_reload: bool = False) -> ad.AnnData:
    """Convenience wrapper around VisiumDataset.load()."""
    return VisiumDataset(cfg).load(force_reload=force_reload)
'''

with open("src/datasets/visium_loader.py", "w") as f:
    f.write(visium_loader_code)

print("Created: src/datasets/visium_loader.py")

Created: src/datasets/visium_loader.py


In [12]:
from src.datasets.visium_loader import load_visium

adata = load_visium(cfg)
print(adata)

Loading cached AnnData from data\processed\adata_raw.h5ad
AnnData object with n_obs × n_vars = 3798 × 36601
    obs: 'in_tissue', 'array_row', 'array_col'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatial'
    obsm: 'spatial'


In [13]:
# sanity check

print("=== Spot metadata (obs) ===")
print(adata.obs.head())

print("\n=== Gene metadata (var) ===")
print(adata.var.head())

print("\n=== Spatial coordinates ===")
print(pd.DataFrame(adata.obsm["spatial"], columns=["x", "y"]).describe())

print("\n=== Count matrix ===")
print("Type :", type(adata.X))
print("Shape:", adata.X.shape)
print("Min  :", adata.X.min())
print("Max  :", adata.X.max())

=== Spot metadata (obs) ===
                    in_tissue  array_row  array_col
AAACAAGTATCTCCCA-1          1         50        102
AAACACCAATAACTGC-1          1         59         19
AAACAGAGCGACTCCT-1          1         14         94
AAACAGGGTCTATATT-1          1         47         13
AAACAGTGTTCCTGGG-1          1         73         43

=== Gene metadata (var) ===
                    gene_ids    feature_types  genome
MIR1302-2HG  ENSG00000243485  Gene Expression  GRCh38
FAM138A      ENSG00000237613  Gene Expression  GRCh38
OR4F5        ENSG00000186092  Gene Expression  GRCh38
AL627309.1   ENSG00000238009  Gene Expression  GRCh38
AL627309.3   ENSG00000239945  Gene Expression  GRCh38

=== Spatial coordinates ===
                  x             y
count   3798.000000   3798.000000
mean   12655.216166  13110.533175
std     4370.339575   4980.331819
min     4176.000000   4046.000000
25%     8838.250000   8801.000000
50%    12778.500000  12958.500000
75%    16467.750000  17590.750000
max   